In [1]:
!pip install -q wandb huggingface_hub

In [2]:
from kaggle_secrets import UserSecretsClient
import wandb
from huggingface_hub import login

# Retrieve secrets
user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login securely
wandb.login(key=wandb_api_key)
login(token=hf_token)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ojasavrathore_25afi13 (ojasavrathore_25afi13-delhi-technological-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torchvision import datasets, transforms
from huggingface_hub import HfApi
import matplotlib.pyplot as plt
import numpy as np
import os
import itertools

In [4]:
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    hf_token = user_secrets.get_secret("HF_TOKEN")
    
    wandb.login(key=wandb_api_key)
    login(token=hf_token)
    print("Successfully authenticated with Weights & Biases and Hugging Face.")
except Exception as e:
    print(f"Authentication Error: {e}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Successfully authenticated with Weights & Biases and Hugging Face.


In [5]:
def get_dataloaders(batch_size):
    transform = transforms.Compose([transforms.ToTensor()])
    
    train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
    test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
    
    full_dataset = ConcatDataset([train_data, test_data])
    total_size = len(full_dataset)
    
    train_size = int(0.8 * total_size)
    val_size = int(0.1 * total_size)
    test_size = total_size - train_size - val_size
    
    train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])
    
    workers = 2 
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, 
                              num_workers=workers, pin_memory=True)
                              
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, 
                            num_workers=workers, pin_memory=True)
                            
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, 
                             num_workers=workers, pin_memory=True)
    
    return train_loader, val_loader, test_loader

In [6]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(28*28, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 28*28),
            nn.Sigmoid() 
        )
        
    def forward(self, x):
        x = x.view(-1, 28*28) 
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        return reconstructed, z

class VAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 28*28),
            nn.Sigmoid()
        )
        
    def encode(self, x):
        h1 = torch.relu(self.fc1(x))
        return self.fc_mu(h1), self.fc_logvar(h1)
        
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
        
    def forward(self, x):
        x = x.view(-1, 28*28) 
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstructed = self.decoder(z)
        return reconstructed, mu, logvar

In [7]:
def vae_loss(recon_x, x, mu, logvar, loss_type="BCE"):
    if loss_type == "BCE":
        recon_loss = nn.functional.binary_cross_entropy(recon_x, x.view(-1, 784), reduction='sum')
    else:
        recon_loss = nn.functional.mse_loss(recon_x, x.view(-1, 784), reduction='sum')
        
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kld, recon_loss, kld

In [8]:
def train_model(config, train_loader, val_loader, test_loader, device):
    run_name = f"{config['model_type']}_dim{config['latent_dim']}_{config['loss_fn']}_{config['optimizer']}"
    print(f"\n{'='*50}\nStarting Run: {run_name}\n{'='*50}")
    
    wandb.init(project="experiment-8-ae-vae", config=config, name=run_name, reinit=True, mode="online")
    
    if config["model_type"] == "AE":
        model = Autoencoder(config["latent_dim"]).to(device)
    else:
        model = VAE(config["latent_dim"]).to(device)
        
    if config["optimizer"] == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"])
    elif config["optimizer"] == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=config["learning_rate"])
    elif config["optimizer"] == "RMSprop":
        optimizer = optim.RMSprop(model.parameters(), lr=config["learning_rate"])
        
    criterion = nn.BCELoss(reduction='sum') if config["loss_fn"] == "BCE" else nn.MSELoss(reduction='sum')

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0
        for batch_idx, (data, _) in enumerate(train_loader):
            data = data.to(device)
            optimizer.zero_grad()
            
            if config["model_type"] == "AE":
                recon, _ = model(data)
                loss = criterion(recon, data.view(-1, 784))
            else:
                recon, mu, logvar = model(data)
                loss, _, _ = vae_loss(recon, data, mu, logvar, config["loss_fn"])
                

            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            train_loss += loss.item()
            optimizer.step()
            
        avg_train_loss = train_loss / len(train_loader.dataset)
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for data, _ in val_loader:
                data = data.to(device)
                if config["model_type"] == "AE":
                    recon, _ = model(data)
                    loss = criterion(recon, data.view(-1, 784))
                else:
                    recon, mu, logvar = model(data)
                    loss, _, _ = vae_loss(recon, data, mu, logvar, config["loss_fn"])
                val_loss += loss.item()
        
        avg_val_loss = val_loss / len(val_loader.dataset)
        
        wandb.log({"train_loss": avg_train_loss, "val_loss": avg_val_loss, "epoch": epoch})
        print(f"Epoch {epoch+1}/{config['epochs']} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    model.eval()
    with torch.no_grad():
        iter_test = iter(test_loader)
        images, _ = next(iter_test)
        img1, img2 = images[0].to(device), images[1].to(device)
        
        if config["model_type"] == "AE":
            _, z1 = model(img1.unsqueeze(0))
            _, z2 = model(img2.unsqueeze(0))
        else:
            mu1, _ = model.encode(img1.view(-1, 28*28))
            mu2, _ = model.encode(img2.view(-1, 28*28))
            z1, z2 = mu1, mu2 

        alphas = np.linspace(0, 1, 10)
        fig, axes = plt.subplots(1, 10, figsize=(15, 2))
        
        for i, alpha in enumerate(alphas):
            z_interp = (1 - alpha) * z1 + alpha * z2
            reconstructed_interp = model.decoder(z_interp).view(28, 28).cpu().numpy()
            axes[i].imshow(reconstructed_interp, cmap='gray')
            axes[i].axis('off')
            
        img_path = f"/kaggle/working/{run_name}_interpolation.png"
        plt.savefig(img_path)
        wandb.log({"Latent Interpolation": wandb.Image(img_path)})
        plt.close(fig)

    model_path = f"/kaggle/working/{run_name}_model.pth"
    torch.save(model.state_dict(), model_path)
    
    wandb.finish()
    
    return run_name, model_path, img_path

In [9]:
def run_all_experiments():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    train_loader, val_loader, test_loader = get_dataloaders(batch_size=64)
    
    model_types = ["AE", "VAE"]
    latent_dims = [2, 8, 16, 32]
    losses = ["BCE", "MSE"]
    optimizers = ["Adam", "RMSprop", "SGD"]
    
    saved_files = []

    for m, dim, loss, opt in itertools.product(model_types, latent_dims, losses, optimizers):
        
        current_config = {
            "model_type": m,
            "latent_dim": dim,
            "batch_size": 64,
            "epochs": 5,
            "learning_rate": 1e-3,
            "optimizer": opt,
            "loss_fn": loss
        }
        
        run_name, model_path, img_path = train_model(
            current_config, train_loader, val_loader, test_loader, device
        )
        
        saved_files.append((run_name, model_path, img_path))
        
    return saved_files

In [10]:
def upload_all_to_huggingface(repo_id, saved_files):
    api = HfApi()
    api.create_repo(repo_id=repo_id, exist_ok=True)
    
    print("\nStarting batch upload to Hugging Face...")
    for run_name, model_path, img_path in saved_files:
        try:
            api.upload_file(
                path_or_fileobj=model_path,
                path_in_repo=f"{run_name}_model.pth",
                repo_id=repo_id
            )
            api.upload_file(
                path_or_fileobj=img_path,
                path_in_repo=f"{run_name}_interpolation.png",
                repo_id=repo_id
            )
            print(f"Uploaded: {run_name}")
        except Exception as e:
            print(f"Failed to upload {run_name}: {e}")

all_results = run_all_experiments()

HF_REPO_ID = "ojasav-rathore/experiment-8-models" 
upload_all_to_huggingface(HF_REPO_ID, all_results)

Using device: cuda


100%|██████████| 26.4M/26.4M [00:01<00:00, 16.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 307kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.66MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 12.8MB/s]



Starting Run: AE_dim2_BCE_Adam


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/5 | Train Loss: 276.6800 | Val Loss: 265.3620
Epoch 2/5 | Train Loss: 260.6074 | Val Loss: 262.4572
Epoch 3/5 | Train Loss: 258.3941 | Val Loss: 261.0953
Epoch 4/5 | Train Loss: 257.2518 | Val Loss: 260.4103
Epoch 5/5 | Train Loss: 256.4527 | Val Loss: 259.6751


...

New Data Upload: |          |  0.00B /  0.00B            

Uploaded: VAE_dim32_MSE_RMSprop


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded: VAE_dim32_MSE_SGD
